
> ## ❌ 결론 — 영상 생체역학 융합: **미채택 (정형 단독 확정)**
>
> 전체 5시즌(영상매칭 3,783경기)으로 **타겟 3종 × 피처선택 × subgroup** 전방위 검증 → 모두 개선 없음.
>
> | 검증 | 결과 |
> |---|---|
> | whiff% 융합 (30 seed) | ΔR² **-0.0092**, p≈9.6e-7 → **유의 악화** |
> | 제구 타겟 (zone%/ball%) | zone% ΔR² -0.0065 (p=0.0006) / ball% 무의 |
> | SHAP 상위 N만 융합 | 전부 무의 (개선 ≈ 0) |
> | subgroup (좌우투·폼변동) | 일관된 기여 없음 |
>
> **왜 안 됐나**: 영상 각도와 타겟의 상관 자체가 약함(\|r\|≈0.02, 정형의 절반). 정형과 중복도 아님(정형→영상 설명 R²=0.10) — 즉 **영상은 다른 정보를 담지만 그게 경기 결과와 무관**. 릴리스 "정지 1프레임"이라 컨디션이 담긴 동작 시퀀스를 못 잡은 것이 근본 한계로 추정.
>
> 상세 수치는 [CLAUDE.md](../../CLAUDE.md) / `4_output/bio_experiment_results.csv`

# 12. 생체역학 Feature 실험

**선행 조건**
- `11_feature_selection.ipynb` 완료 → `4_output/statcast_final_model.pkl`, `4_output/final_feature_cols.csv`
- 영상 파이프라인 완료 → `0_data/4_features/features_bio_pitch15.parquet`

**목적**: 정형 Statcast feature에 영상 기반 생체역학 feature를 추가했을 때 예측력이 향상되는지 검증

| 실험 | 내용 |
|---|---|
| E7-1 | 정형 feature만 (기준, Phase 8 최종 모델 재확인) |
| E7-2 | 생체역학 feature만 |
| E7-3 | 정형 + 생체역학 전체 융합 |
| E7-4 | 융합 모델 SHAP — 생체역학 feature 중요도 |
| E7-5 | 정형 vs 정형+생체역학 paired t-test (n=30 seeds) |
| E7-6 | 최적 feature set 확정 + 최종 융합 모델 저장 |

In [14]:
import os, sys

IN_COLAB = os.path.exists('/content')

if IN_COLAB:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    DRIVE   = '/content/drive/MyDrive/MLB_pitcher'
    DATA    = os.path.join(DRIVE, 'data')
    BIO_DIR = os.path.join(DATA, '4_features')   # 11번이 만든 bio parquet
    # 정형 parquet 위치: features / 4_features 중 존재하는 쪽 자동 선택
    STAT_DIR = next((d for d in [os.path.join(DATA, 'features'),
                                 os.path.join(DATA, '4_features'),
                                 BIO_DIR]
                     if os.path.exists(os.path.join(d, 'features_pitch15.parquet'))),
                    os.path.join(DATA, 'features'))
    OUTPUT_DIR = os.path.join(DRIVE, '4_output')
else:
    DRIVE    = r'c:\Users\suyou\OneDrive\Desktop\ASAC\PROJECT\투수 컨디션 예측'
    BIO_DIR  = os.path.join(DRIVE, '0_data', '4_features')
    STAT_DIR = os.path.join(DRIVE, '0_data', '4_features')
    OUTPUT_DIR = os.path.join(DRIVE, '4_output')

STATCAST_PATH = os.path.join(STAT_DIR, 'features_pitch15.parquet')
BIO_PATH      = os.path.join(BIO_DIR,  'features_bio_pitch15_full9.parquet')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'환경: {"코랩" if IN_COLAB else "로컬"}')
print(f'Statcast : {STATCAST_PATH}  (존재: {os.path.exists(STATCAST_PATH)})')
print(f'Bio      : {BIO_PATH}  (존재: {os.path.exists(BIO_PATH)})')

if not os.path.exists(STATCAST_PATH):
    raise FileNotFoundError(f'정형 피처 파일 없음: {STATCAST_PATH}')
if not os.path.exists(BIO_PATH):
    raise FileNotFoundError(f'영상 피처 파일 없음: {BIO_PATH} — 11번 먼저 실행')


환경: 코랩
Statcast : /content/drive/MyDrive/MLB_pitcher/data/4_features/features_pitch15.parquet  (존재: True)
Bio      : /content/drive/MyDrive/MLB_pitcher/data/4_features/features_bio_pitch15_mean_std.parquet  (존재: True)


In [15]:
try:
    import xgboost, shap
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'xgboost', 'shap', '-q'])

import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import joblib
from sklearn.metrics import mean_squared_error, r2_score
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('모듈 로드 완료')

모듈 로드 완료


## ★ 선행 단계 완료 후 최적 파라미터 입력 (필수)

In [16]:
# 10_tuning_experiment 결과의 최적 파라미터로 교체
BEST_PARAMS = {
    'n_estimators':     300,    # ← Phase 7 결과로 교체
    'learning_rate':    0.05,
    'max_depth':        6,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'reg_alpha':        0.0,
    'reg_lambda':       1.0,
    'random_state': 42, 'n_jobs': -1, 'verbosity': 0,
    'early_stopping_rounds': 50,
}
print('파라미터 설정 완료')

파라미터 설정 완료


## 1. 데이터 로드 및 병합

In [18]:
META_COLS = ['game_pk', 'pitcher', 'season', 'y_whiff']
MERGE_KEYS = ['game_pk', 'pitcher', 'season']

df_stat = pd.read_parquet(STATCAST_PATH)
df_bio  = pd.read_parquet(BIO_PATH)

# bio 쪽 비피처 컬럼 정리: y_whiff(있으면)·집계메타(n_pitches_used) 제거
DROP_FROM_BIO = ['y_whiff', 'n_pitches_used']
df_bio = df_bio.drop(columns=[c for c in DROP_FROM_BIO if c in df_bio.columns])

print(f'Statcast  : {df_stat.shape}  columns={len(df_stat.columns)}')
print(f'Bio       : {df_bio.shape}   columns={len(df_bio.columns)}')

# 키 컬럼으로 병합 (bio엔 이미 game_pk/pitcher/season 있음 → 중복 추가 X)
df = df_stat.merge(df_bio, on=MERGE_KEYS, how='inner')
print(f'\n병합 후  : {df.shape}  (정형 {len(df_stat)}행 중 영상매칭 {len(df)}행)')

STAT_COLS = [c for c in df_stat.columns if c not in META_COLS]
BIO_COLS  = [c for c in df_bio.columns  if c not in MERGE_KEYS]   # 순수 영상 피처
ALL_COLS  = STAT_COLS + BIO_COLS

train = df[df['season'].isin([2021, 2022, 2023])]
val   = df[df['season'] == 2024]
test  = df[df['season'] == 2025]

print(f'\nTrain: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}')
print(f'Statcast features: {len(STAT_COLS)}  Bio features: {len(BIO_COLS)}')
print(f'Bio 피처: {BIO_COLS}')


Statcast  : (23225, 63)  columns=63
Bio       : (1278, 22)   columns=22
Bio columns: ['pitcher_id', 'stride_norm_mean', 'arm_slot_mean', 'shoulder_tilt_mean', 'hip_tilt_mean', 'trunk_dist_norm_mean', 'trunk_angle_mean', 'separation_mean', 'release_height_norm_mean', 'arm_extension_norm_mean', 'stride_norm_std', 'arm_slot_std', 'shoulder_tilt_std', 'hip_tilt_std', 'trunk_dist_norm_std', 'trunk_angle_std', 'separation_std', 'release_height_norm_std', 'arm_extension_norm_std', 'n_pitches_used']


KeyError: "['pitcher'] not in index"

## 2. 단독/융합 비교

In [ ]:
def train_eval(feat_cols, label=''):
    X_tr = train[feat_cols]; y_tr = train['y_whiff']
    X_vl = val[feat_cols];   y_vl = val['y_whiff']
    X_te = test[feat_cols];  y_te = test['y_whiff']

    m = xgb.XGBRegressor(**BEST_PARAMS)
    m.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)

    vr2    = r2_score(y_vl, m.predict(X_vl))
    vrmse  = mean_squared_error(y_vl, m.predict(X_vl)) ** 0.5
    tr2    = r2_score(y_te, m.predict(X_te))
    trmse  = mean_squared_error(y_te, m.predict(X_te)) ** 0.5
    print(f'[{label}] feat={len(feat_cols)}  Val R²={vr2:.4f} RMSE={vrmse:.4f} | Test R²={tr2:.4f} RMSE={trmse:.4f}')
    return m, vr2, vrmse, tr2, trmse

results = []
model_stat, vr2, vrmse, tr2, trmse = train_eval(STAT_COLS, 'E7-1 정형만')
results.append({'name': 'E7-1 정형만', 'n_feat': len(STAT_COLS), 'val_r2': vr2, 'val_rmse': vrmse, 'test_r2': tr2, 'test_rmse': trmse})

model_bio, vr2, vrmse, tr2, trmse = train_eval(BIO_COLS, 'E7-2 생체역학만')
results.append({'name': 'E7-2 생체역학만', 'n_feat': len(BIO_COLS), 'val_r2': vr2, 'val_rmse': vrmse, 'test_r2': tr2, 'test_rmse': trmse})

model_fused, vr2, vrmse, tr2, trmse = train_eval(ALL_COLS, 'E7-3 융합')
results.append({'name': 'E7-3 융합', 'n_feat': len(ALL_COLS), 'val_r2': vr2, 'val_rmse': vrmse, 'test_r2': tr2, 'test_rmse': trmse})

res_df = pd.DataFrame(results)
print('\n=== 결과 요약 ===')
print(res_df.round(4).to_string(index=False))

## 3. 융합 모델 SHAP — 생체역학 기여도

In [ ]:
X_val_all = val[ALL_COLS]
explainer  = shap.TreeExplainer(model_fused)
shap_vals  = explainer(X_val_all)
shap_mean  = pd.Series(np.abs(shap_vals.values).mean(axis=0), index=ALL_COLS).sort_values(ascending=False)

# 생체역학 feature 순위 확인
bio_ranks = {feat: (i+1) for i, feat in enumerate(shap_mean.index) if feat in BIO_COLS}
print(f'\n생체역학 feature SHAP 순위 (총 {len(ALL_COLS)}개 중):')
for feat, rank in sorted(bio_ranks.items(), key=lambda x: x[1]):
    print(f'  {rank:3d}위  {feat}  (SHAP={shap_mean[feat]:.5f})')

stat_shap = shap_mean[STAT_COLS].mean()
bio_shap  = shap_mean[BIO_COLS].mean()
print(f'\n정형 feature 평균 SHAP : {stat_shap:.5f}')
print(f'생체역학 feature 평균 SHAP: {bio_shap:.5f}')

# 시각화 (shap.summary_plot은 ax 미지원 → pandas barh로 직접)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 좌: 전체 SHAP Top 30 (영상 피처는 주황으로 강조)
top = shap_mean.head(30)[::-1]
colors = ['darkorange' if f in BIO_COLS else 'steelblue' for f in top.index]
axes[0].barh(range(len(top)), top.values, color=colors)
axes[0].set_yticks(range(len(top)))
axes[0].set_yticklabels(top.index, fontsize=8)
axes[0].set_xlabel('mean |SHAP|')
axes[0].set_title('융합 모델 SHAP Top 30 (주황=영상)')

# 우: 정형 vs 영상 평균 기여도
type_df = pd.DataFrame({'유형': ['정형(Statcast)', '생체역학(영상)'],
                        '평균 SHAP': [stat_shap, bio_shap]})
axes[1].bar(type_df['유형'], type_df['평균 SHAP'], color=['steelblue', 'darkorange'])
axes[1].set_ylabel('평균 |SHAP|')
axes[1].set_title('정형 vs 생체역학 feature 평균 기여도')
for i, v in enumerate(type_df['평균 SHAP']):
    axes[1].text(i, v, f'{v:.5f}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'shap_bio_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()


## 4. Paired t-test — 정형 vs 정형+생체역학

In [ ]:
N_SEEDS = 30
r2_stat_list  = []
r2_fused_list = []

for seed in range(N_SEEDS):
    p = {**BEST_PARAMS, 'random_state': seed}

    m_s = xgb.XGBRegressor(**p)
    m_s.fit(train[STAT_COLS], train['y_whiff'],
            eval_set=[(val[STAT_COLS], val['y_whiff'])], verbose=False)
    r2_stat_list.append(r2_score(val['y_whiff'], m_s.predict(val[STAT_COLS])))

    m_f = xgb.XGBRegressor(**p)
    m_f.fit(train[ALL_COLS], train['y_whiff'],
            eval_set=[(val[ALL_COLS], val['y_whiff'])], verbose=False)
    r2_fused_list.append(r2_score(val['y_whiff'], m_f.predict(val[ALL_COLS])))

    if (seed + 1) % 10 == 0:
        print(f'seed {seed+1}/{N_SEEDS} 완료  '
              f'정형={np.mean(r2_stat_list):.4f}  '
              f'융합={np.mean(r2_fused_list):.4f}')

t_stat, p_val = stats.ttest_rel(r2_fused_list, r2_stat_list)
diff = np.mean(r2_fused_list) - np.mean(r2_stat_list)

print(f'\n=== Paired t-test 결과 ===')
print(f'정형만  : mean R²={np.mean(r2_stat_list):.4f} ± {np.std(r2_stat_list):.4f}')
print(f'융합    : mean R²={np.mean(r2_fused_list):.4f} ± {np.std(r2_fused_list):.4f}')
print(f'차이    : {diff:+.4f}')
print(f't={t_stat:.3f}  p={p_val:.4f}  {"✅ 유의 (p<0.05)" if p_val < 0.05 else "❌ 유의하지 않음"}')

## 6. 영상 SHAP 상위 N개만 선택 → 재실험

full9(81피처)는 과적합. SHAP 상위 영상 피처만 골라 노이즈 제거 후 정형과 융합.
`TOP_BIO`(10/15/20)를 바꿔 비교. shap_mean(cell11에서 계산됨)이 메모리에 있어야 함.

In [ ]:
from scipy import stats

# cell11의 shap_mean(전체 피처 SHAP) 기준으로 영상 피처만 순위 추출
bio_ranked = [f for f in shap_mean.index if f in BIO_COLS]

def eval_topbio(top_n, n_seeds=30):
    sel_bio = bio_ranked[:top_n]
    cols = STAT_COLS + sel_bio
    r2_stat, r2_fused = [], []
    for seed in range(n_seeds):
        prm = {**BEST_PARAMS, 'random_state': seed}
        ms = xgb.XGBRegressor(**prm)
        ms.fit(train[STAT_COLS], train['y_whiff'],
               eval_set=[(val[STAT_COLS], val['y_whiff'])], verbose=False)
        r2_stat.append(r2_score(val['y_whiff'], ms.predict(val[STAT_COLS])))
        mf = xgb.XGBRegressor(**prm)
        mf.fit(train[cols], train['y_whiff'],
               eval_set=[(val[cols], val['y_whiff'])], verbose=False)
        r2_fused.append(r2_score(val['y_whiff'], mf.predict(val[cols])))
    t, pv = stats.ttest_rel(r2_fused, r2_stat)
    return np.mean(r2_stat), np.mean(r2_fused), np.mean(r2_fused)-np.mean(r2_stat), pv, sel_bio

print('=== 영상 SHAP 상위 N개 선택 융합 (정형 vs 정형+상위N) ===')
rows = []
for top_n in [5, 10, 15, 20]:
    rs, rf, diff, pv, sel = eval_topbio(top_n)
    mark = '✅유의' if pv < 0.05 else '❌무의'
    print(f'  TOP {top_n:2d}: 정형={rs:.4f}  융합={rf:.4f}  차이={diff:+.4f}  p={pv:.4f}  {mark}')
    rows.append({'top_n': top_n, 'stat_r2': rs, 'fused_r2': rf, 'diff': diff, 'p': pv})

top_df = pd.DataFrame(rows)
top_df.to_csv(os.path.join(OUTPUT_DIR, 'bio_topN_results.csv'), index=False)

# 가장 좋은 구성의 선택 피처 출력
best = top_df.loc[top_df['fused_r2'].idxmax()]
print(f'\n최고 융합 R²: TOP {int(best.top_n)} (R²={best.fused_r2:.4f}, p={best.p:.4f})')
_,_,_,_,best_sel = eval_topbio(int(best.top_n), n_seeds=1)
print('선택된 영상 피처:', best_sel)


## 5. 최적 feature set 확정 및 저장

In [ ]:
# ★ 위 실험 결과 보고 결정 — 기본: 융합(All)
USE_BIO = True   # False면 정형만 사용

FINAL_COLS = ALL_COLS if USE_BIO else STAT_COLS
X_tr = train[FINAL_COLS]; y_tr = train['y_whiff']
X_vl = val[FINAL_COLS];   y_vl = val['y_whiff']
X_te = test[FINAL_COLS];  y_te = test['y_whiff']

final_model = xgb.XGBRegressor(**BEST_PARAMS)
final_model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)

print(f'최종 구성: {"정형+생체역학" if USE_BIO else "정형만"} ({len(FINAL_COLS)} features)')
print(f'Val  R²={r2_score(y_vl, final_model.predict(X_vl)):.4f}  RMSE={mean_squared_error(y_vl, final_model.predict(X_vl))**0.5:.4f}')
print(f'Test R²={r2_score(y_te, final_model.predict(X_te)):.4f}  RMSE={mean_squared_error(y_te, final_model.predict(X_te))**0.5:.4f}')

model_fname = 'fused_final_model.pkl' if USE_BIO else 'statcast_final_model.pkl'
joblib.dump(final_model, os.path.join(OUTPUT_DIR, model_fname))
pd.Series(FINAL_COLS).to_csv(os.path.join(OUTPUT_DIR, 'fused_feature_cols.csv' if USE_BIO else 'final_feature_cols.csv'), index=False)

res_df_ext = pd.concat([res_df, pd.DataFrame([{
    'name': f'E7-5 paired t-test',
    'n_feat': '—',
    'val_r2': f'stat={np.mean(r2_stat_list):.4f} / fused={np.mean(r2_fused_list):.4f}',
    'val_rmse': f'p={p_val:.4f}',
    'test_r2': '—', 'test_rmse': '—'
}])], ignore_index=True)
res_df_ext.to_csv(os.path.join(OUTPUT_DIR, 'bio_experiment_results.csv'), index=False)

print(f'\n저장 완료: {model_fname} / bio_experiment_results.csv')

## 7. E8 제구(control) 타겟으로 영상 융합 재검증

whiff%(공의 위력)엔 영상이 무의였으나, **제구 지표**는 릴리스 일관성과 인과가 더 가까울 수 있음.
새 타겟 3종(`zone%`·`ball%`·`BB%`)을 만들어 정형 vs 융합을 다시 검증.

- **선행**: 아래 [] 셀로 `game_control_targets.parquet` 먼저 생성 (Statcast 원본에서 새 Y 계산).
- **로직**: ① 정형 baseline으로 예측 가능한 타겟인지 스크리닝(whiff 0.036 수준) → ② 통과한 타겟만 영상 융합 30-seed paired t-test.
- **해석**: 🎯 하나라도 개선되면 "타겟-피처 인과 정렬로 영상이 살아남"(반전), 전부 ❌면 "제구에서도 무의 = 영상 각도 자체의 한계 확정".

In [ ]:
# [E8-0] 제구 타겟 생성 — zone% / ball% / BB% (한 번만 실행하면 parquet 저장됨)
import glob

if IN_COLAB:
    STATCAST_GLOB = f'{DRIVE}/statcast/statcast_*.parquet'
    CTRL_OUT = f'{DRIVE}/data/game_control_targets.parquet'
else:
    STATCAST_GLOB = os.path.join(DRIVE, '0_data', 'statcast', 'statcast_*.parquet')
    CTRL_OUT = os.path.join(DRIVE, '0_data', 'data', 'game_control_targets.parquet')

X_PITCHES, MIN_PITCHES_Y, MIN_Y_BF = 15, 20, 5   # 04_build_targets와 동일 규칙

if os.path.exists(CTRL_OUT):
    print('이미 존재:', CTRL_OUT, '— 재생성하려면 파일 삭제 후 실행')
else:
    USE_COLS = ['game_pk','pitcher','game_year','at_bat_number','pitch_number',
                'events','description','type','zone']
    files = sorted(glob.glob(STATCAST_GLOB)); print('statcast 파일:', len(files))
    dfs = []
    for f in files:
        d = pd.read_parquet(f); d = d[[c for c in USE_COLS if c in d.columns]].copy()
        if 'game_year' in d.columns: d = d.rename(columns={'game_year':'season'})
        dfs.append(d)
    sc = pd.concat(dfs, ignore_index=True)
    sc = sc.dropna(subset=['game_pk','pitcher','at_bat_number','pitch_number'])
    sc[['game_pk','pitcher','at_bat_number','pitch_number']] = \
        sc[['game_pk','pitcher','at_bat_number','pitch_number']].astype('int64')

    sc = sc.sort_values(['game_pk','pitcher','at_bat_number','pitch_number'])
    sc['pitch_rank'] = sc.groupby(['game_pk','pitcher']).cumcount() + 1
    yz = sc[sc['pitch_rank'] > X_PITCHES].copy()      # Y구간(16구~)

    z = pd.to_numeric(yz['zone'], errors='coerce')
    yz['_in_zone'] = z.between(1, 9).fillna(False).astype(int)   # 존 1~9 = 안 (NA→0)
    yz['_is_ball'] = (yz['type'].astype(str) == 'B').astype(int)
    ev = yz['events'].fillna('')
    yz['_bb'] = ev.isin(['walk','intent_walk']).astype(int)
    yz['_bf'] = (ev != '').astype(int)                # 타석 종료 = 타자수

    KEYS = ['game_pk','pitcher','season']
    agg = yz.groupby(KEYS).agg(
        n_pitch=('_in_zone','size'), in_zone=('_in_zone','sum'),
        balls=('_is_ball','sum'), bb=('_bb','sum'), bf=('_bf','sum')).reset_index()
    hi = agg[agg['n_pitch'] >= MIN_PITCHES_Y].copy()
    hi['y_zone_rate'] = (hi['in_zone']/hi['n_pitch']).round(4)   # 존 적중률↑=제구 좋음
    hi['y_ball_rate'] = (hi['balls']/hi['n_pitch']).round(4)     # 볼 비율↑=제구 나쁨
    hi = hi[hi['bf'] >= MIN_Y_BF].copy()
    hi['y_bb_rate'] = (hi['bb']/hi['bf']).round(4)              # 볼넷율↑=제구 나쁨

    ctrl_targets = hi[KEYS + ['y_zone_rate','y_ball_rate','y_bb_rate']].copy()
    ctrl_targets.to_parquet(CTRL_OUT, index=False)
    print('저장:', CTRL_OUT, '| shape:', ctrl_targets.shape)
    print(ctrl_targets[['y_zone_rate','y_ball_rate','y_bb_rate']].describe().round(3).T.to_string())
    # 참고: MLB 평균 zone%~0.48, ball%~0.36, BB%~0.08 근처면 계산 정상

In [ ]:
# [E8-1] 제구 타겟으로 정형 vs 융합 재검증 (df/STAT_COLS/BIO_COLS/ALL_COLS 재사용)
ctrl = pd.read_parquet(CTRL_OUT)
dfx = df.merge(ctrl, on=['game_pk','pitcher','season'], how='inner')
print(f'융합df {len(df)}행 + 제구타겟 → 병합 {len(dfx)}행')
CTRL_TARGETS = ['y_zone_rate', 'y_ball_rate', 'y_bb_rate']

def _split(fr):
    return (fr[fr['season'].isin([2021,2022,2023])],
            fr[fr['season']==2024], fr[fr['season']==2025])

def _r2(cols, ycol, tr, vl):
    m = xgb.XGBRegressor(**{**BEST_PARAMS, 'random_state': 0})
    m.fit(tr[cols], tr[ycol], eval_set=[(vl[cols], vl[ycol])], verbose=False)
    return r2_score(vl[ycol], m.predict(vl[cols]))

# --- 1단계: 정형 baseline 스크리닝 ---
print('\n[1단계] 정형 baseline Val R² (whiff=0.036 수준 나오는 타겟 찾기)')
tr, vl, te = _split(dfx); screen = {}
for y in CTRL_TARGETS:
    r2 = _r2(STAT_COLS, y, tr.dropna(subset=[y]), vl.dropna(subset=[y]))
    screen[y] = r2
    print(f'  {y:14s}: 정형 Val R²={r2:+.4f}  {"OK 예측가능" if r2 > 0.01 else "X 노이즈(정형도 안됨)"}')

# --- 2단계: baseline>0.01 인 타겟만 융합 paired t-test(30 seed) ---
print('\n[2단계] 정형 vs 융합 paired t-test')
def _ptest(ycol, n=30):
    sub = dfx.dropna(subset=[ycol]); tr, vl, te = _split(sub); rs, rf = [], []
    for s in range(n):
        p = {**BEST_PARAMS, 'random_state': s}
        ms = xgb.XGBRegressor(**p)
        ms.fit(tr[STAT_COLS], tr[ycol], eval_set=[(vl[STAT_COLS], vl[ycol])], verbose=False)
        rs.append(r2_score(vl[ycol], ms.predict(vl[STAT_COLS])))
        mf = xgb.XGBRegressor(**p)
        mf.fit(tr[ALL_COLS], tr[ycol], eval_set=[(vl[ALL_COLS], vl[ycol])], verbose=False)
        rf.append(r2_score(vl[ycol], mf.predict(vl[ALL_COLS])))
    t, pv = stats.ttest_rel(rf, rs)
    return np.mean(rs), np.mean(rf), np.mean(rf)-np.mean(rs), t, pv

ctrl_rows = []
for y in CTRL_TARGETS:
    if screen[y] <= 0.01:
        print(f'  {y:14s}: baseline<0.01 → 융합 스킵(무대 없음)'); continue
    rs, rf, d, t, pv = _ptest(y)
    mark = 'GOAL 유의 개선!' if (pv < 0.05 and d > 0) else ('유의 악화' if pv < 0.05 else '무의')
    print(f'  {y:14s}: 정형={rs:.4f} 융합={rf:.4f} 차이={d:+.4f} t={t:.2f} p={pv:.4f}  {mark}')
    ctrl_rows.append({'target': y, 'stat_r2': rs, 'fused_r2': rf, 'diff': d, 't': t, 'p': pv})

if ctrl_rows:
    pd.DataFrame(ctrl_rows).to_csv(os.path.join(OUTPUT_DIR, 'control_fusion_results.csv'), index=False)
    print('\n저장: control_fusion_results.csv')
print('\n[해석] 하나라도 "유의 개선" → 영상이 제구 타겟에서 살아남(반전) / 전부 무의 → 영상 한계 확정')

In [ ]:
# [E8-2] 제구 타겟에 영상이 왜 안 되는지 진단 + SHAP 상위N 재검증
#  (E8-1 다음 — dfx / STAT_COLS / BIO_COLS / ALL_COLS / BEST_PARAMS 재사용)
import shap

# ── A. 영상 각도 vs 제구 타겟 상관 (신호 자체 있나?) ──────────────
print('='*64)
print('[A] 영상 피처 vs 제구 타겟 상관 (|r|) — 신호가 있나?')
print('='*64)
for y in ['y_zone_rate', 'y_ball_rate']:
    sub = dfx.dropna(subset=[y])
    bio_r  = sub[BIO_COLS].corrwith(sub[y]).abs()
    stat_r = sub[STAT_COLS].corrwith(sub[y]).abs()
    print(f'\n[{y}]')
    print(f'  영상 |r| 평균={bio_r.mean():.4f}  최대={bio_r.max():.4f} ({bio_r.idxmax()})')
    print(f'  정형 |r| 평균={stat_r.mean():.4f}  최대={stat_r.max():.4f} ({stat_r.idxmax()})')
    print('  영상 상관 상위3:', bio_r.sort_values(ascending=False).head(3).round(4).to_dict())

# ── B. SHAP 상위 N개 영상 피처만 골라 융합 (노이즈 제거하면 살아나나?) ─
print('\n' + '='*64)
print('[B] 영상 SHAP 상위 N개만 융합 (전체 넣으면 노이즈 → 소수만 골라 재검증)')
print('='*64)

def _split(fr):
    return (fr[fr['season'].isin([2021,2022,2023])],
            fr[fr['season']==2024], fr[fr['season']==2025])

def _topN_ptest(ycol, bio_ranked, top_n, n=30):
    cols = STAT_COLS + bio_ranked[:top_n]
    sub = dfx.dropna(subset=[ycol]); tr, vl, te = _split(sub); rs, rf = [], []
    for s in range(n):
        p = {**BEST_PARAMS, 'random_state': s}
        ms = xgb.XGBRegressor(**p)
        ms.fit(tr[STAT_COLS], tr[ycol], eval_set=[(vl[STAT_COLS], vl[ycol])], verbose=False)
        rs.append(r2_score(vl[ycol], ms.predict(vl[STAT_COLS])))
        mf = xgb.XGBRegressor(**p)
        mf.fit(tr[cols], tr[ycol], eval_set=[(vl[cols], vl[ycol])], verbose=False)
        rf.append(r2_score(vl[ycol], mf.predict(vl[cols])))
    t, pv = stats.ttest_rel(rf, rs)
    return np.mean(rs), np.mean(rf), np.mean(rf)-np.mean(rs), t, pv

for y in ['y_zone_rate', 'y_ball_rate']:
    sub = dfx.dropna(subset=[y]); tr, vl, te = _split(sub)
    m = xgb.XGBRegressor(**{**BEST_PARAMS, 'random_state': 0})
    m.fit(tr[ALL_COLS], tr[y], eval_set=[(vl[ALL_COLS], vl[y])], verbose=False)
    sv = shap.TreeExplainer(m)(vl[ALL_COLS])
    shap_mean = pd.Series(np.abs(sv.values).mean(0), index=ALL_COLS).sort_values(ascending=False)
    bio_ranked = [f for f in shap_mean.index if f in BIO_COLS]
    print(f'\n[{y}]  (영상 SHAP 상위3: {bio_ranked[:3]})')
    for top_n in [3, 5, 10]:
        rs, rf, d, t, pv = _topN_ptest(y, bio_ranked, top_n)
        mark = 'GOAL 유의 개선!' if (pv<0.05 and d>0) else ('유의 악화' if pv<0.05 else '무의')
        print(f'  TOP {top_n:2d}: 정형={rs:.4f} 융합={rf:.4f} 차이={d:+.4f} p={pv:.4f}  {mark}')

print('\n[결과] A: 영상|r|≈0.02~0.08(정형보다 작음) / B: 상위N도 전부 무의')
print('       → "영상 각도는 제구와도 상관 없음, 소수만 골라도 안 됨" = 완전 종결')

In [ ]:
# [E8-3] Subgroup 분석 — "영상이 어디서 기여하나" (팀원 성은 제안)
#  핵심질문: 정형만으로 예측이 빗나가는 구간/투수를 영상이 설명하는가?
#  (E8-2 다음 — dfx / STAT_COLS / BIO_COLS / ALL_COLS / BEST_PARAMS 재사용)
TARGET = 'y_zone_rate'   # 바꿔가며: y_zone_rate / y_ball_rate
sub = dfx.dropna(subset=[TARGET]).copy()
tr, vl, te = _split(sub)

def _fit_pred(cols, tr, ev):
    m = xgb.XGBRegressor(**{**BEST_PARAMS, 'random_state': 0})
    m.fit(tr[cols], tr[TARGET], eval_set=[(vl[cols], vl[TARGET])], verbose=False)
    return m.predict(ev[cols])

ev = vl.copy()
ev['_pred_stat']  = _fit_pred(STAT_COLS, tr, vl)
ev['_pred_fused'] = _fit_pred(ALL_COLS,  tr, vl)
ev['_err_stat']   = (ev[TARGET] - ev['_pred_stat']).abs()

def _sub_r2(mask, name):
    g = ev[mask]
    if len(g) < 30: return None   # 표본 30 미만 스킵
    return {'group': name, 'n': len(g),
            'stat_r2': round(r2_score(g[TARGET], g['_pred_stat']),4),
            'fused_r2': round(r2_score(g[TARGET], g['_pred_fused']),4),
            'diff': round(r2_score(g[TARGET], g['_pred_fused'])-r2_score(g[TARGET], g['_pred_stat']),4)}

rows = []
# 1. 좌/우투
if 'hand' in ev.columns:
    for h in ev['hand'].dropna().unique():
        rows.append(_sub_r2(ev['hand']==h, f'hand={h}'))
# 2. 폼 변동성 큰/작은 투수 (train 중앙값 분할 = leakage 방지)
for feat in ['separation_std','arm_slot_std','release_height_norm_std','trunk_angle_std']:
    if feat in ev.columns:
        med = tr[feat].median()
        rows.append(_sub_r2(ev[feat] >  med, f'{feat}>median(high변동)'))
        rows.append(_sub_r2(ev[feat] <= med, f'{feat}<=median(low변동)'))
# 3. 정형 예측오차 큰/작은 경기
q = ev['_err_stat'].quantile(0.5)
rows.append(_sub_r2(ev['_err_stat'] >  q, 'stat오차>median(정형 틀린 경기)'))
rows.append(_sub_r2(ev['_err_stat'] <= q, 'stat오차<=median(정형 맞은 경기)'))

res = pd.DataFrame([r for r in rows if r]).sort_values('diff', ascending=False)
print(f'=== [{TARGET}] Subgroup별 정형 vs 융합 R² (val, n>=30) ===\n')
print(res.to_string(index=False))
print('\n[해석] diff>0 그룹이 정형R²≈0인 곳뿐이면 우연 / 폼변동 high에서 악화면 가설 반증')
print('       → 어느 subgroup에서도 영상 일관 기여 없음 = 완전 종결')